# Hermes Session Database Probe

这个 notebook 用于查看 Hermes 保存到 SQLite 数据库中的 session 内容。

数据库位置: `~/.hermes/state.db`

In [1]:
import sqlite3
import json
from pathlib import Path
from datetime import datetime

# 获取数据库路径
import sys
sys.path.insert(0, str(Path.cwd().parent))
from hermes_constants import get_hermes_home

db_path = get_hermes_home() / "state.db"
print(f"数据库路径: {db_path}")
print(f"数据库存在: {db_path.exists()}")

数据库路径: /home/uih/.hermes/state.db
数据库存在: True


In [2]:
# 连接数据库
conn = sqlite3.connect(
    str(db_path),
    check_same_thread=False,
    timeout=5.0,
)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
print("✓ 数据库连接成功")

✓ 数据库连接成功


## 1. 数据库表结构

In [3]:
# 查看所有表
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = [row[0] for row in cursor.fetchall()]
print("数据库中的表:")
for t in tables:
    print(f"  - {t}")

数据库中的表:
  - schema_version
  - sessions
  - messages
  - sqlite_sequence
  - messages_fts
  - messages_fts_data
  - messages_fts_idx
  - messages_fts_docsize
  - messages_fts_config


In [4]:
# sessions 表结构
print("\nsessions 表结构:")
cursor.execute("PRAGMA table_info(sessions)")
for row in cursor.fetchall():
    print(f"  {row['name']:25} {row['type']:10} {'NOT NULL' if row['notnull'] else ''}")


sessions 表结构:
  id                        TEXT       
  source                    TEXT       NOT NULL
  user_id                   TEXT       
  model                     TEXT       
  model_config              TEXT       
  system_prompt             TEXT       
  parent_session_id         TEXT       
  started_at                REAL       NOT NULL
  ended_at                  REAL       
  end_reason                TEXT       
  message_count             INTEGER    
  tool_call_count           INTEGER    
  input_tokens              INTEGER    
  output_tokens             INTEGER    
  cache_read_tokens         INTEGER    
  cache_write_tokens        INTEGER    
  reasoning_tokens          INTEGER    
  billing_provider          TEXT       
  billing_base_url          TEXT       
  billing_mode              TEXT       
  estimated_cost_usd        REAL       
  actual_cost_usd           REAL       
  cost_status               TEXT       
  cost_source               TEXT       
  pricing

In [5]:
# messages 表结构
print("\nmessages 表结构:")
cursor.execute("PRAGMA table_info(messages)")
for row in cursor.fetchall():
    print(f"  {row['name']:25} {row['type']:10} {'NOT NULL' if row['notnull'] else ''}")


messages 表结构:
  id                        INTEGER    
  session_id                TEXT       NOT NULL
  role                      TEXT       NOT NULL
  content                   TEXT       
  tool_call_id              TEXT       
  tool_calls                TEXT       
  tool_name                 TEXT       
  timestamp                 REAL       NOT NULL
  token_count               INTEGER    
  finish_reason             TEXT       
  reasoning                 TEXT       
  reasoning_details         TEXT       
  codex_reasoning_items     TEXT       


## 2. Session 列表概览

In [6]:
# 查看所有 session 概览
cursor.execute("""
    SELECT 
        id,
        source,
        title,
        model,
        started_at,
        ended_at,
        message_count,
        tool_call_count,
        parent_session_id
    FROM sessions 
    ORDER BY started_at DESC
""")

sessions = cursor.fetchall()
print(f"共有 {len(sessions)} 个 sessions\n")

for s in sessions[:20]:  # 只显示前20个
    start_time = datetime.fromtimestamp(s['started_at']).strftime('%Y-%m-%d %H:%M:%S') if s['started_at'] else 'N/A'
    end_time = datetime.fromtimestamp(s['ended_at']).strftime('%H:%M:%S') if s['ended_at'] else 'active'
    title = s['title'] or '(untitled)'
    print(f"ID: {s['id'][:30]}...")
    print(f"  来源: {s['source']:<10} 模型: {s['model'] or 'N/A'}")
    print(f"  标题: {title[:50]}")
    print(f"  时间: {start_time} → {end_time}")
    print(f"  消息: {s['message_count']}  工具调用: {s['tool_call_count']}")
    if s['parent_session_id']:
        print(f"  父session: {s['parent_session_id'][:30]}...")
    print()

共有 4 个 sessions

ID: 20260410_113250_ef3a2e...
  来源: cli        模型: glm-5
  标题: (untitled)
  时间: 2026-04-10 11:32:55 → active
  消息: 2  工具调用: 0

ID: 20260410_113008_a5201b...
  来源: cli        模型: glm-5
  标题: (untitled)
  时间: 2026-04-10 11:30:11 → 11:32:47
  消息: 8  工具调用: 1

ID: 20260410_112806_fb9963...
  来源: cli        模型: glm-5
  标题: (untitled)
  时间: 2026-04-10 11:28:09 → 11:28:23
  消息: 1  工具调用: 0

ID: 20260410_104848_9d5e56...
  来源: cli        模型: N/A
  标题: (untitled)
  时间: 2026-04-10 10:48:52 → 11:13:43
  消息: 1  工具调用: 0



## 3. 查看特定 Session 详情

修改下面的 session_id 来查看特定 session 的完整信息。

In [7]:
# 选择要查看的 session（默认最新的一个）
if sessions:
    target_session_id = sessions[0]['id']
    print(f"查看 session: {target_session_id}")
else:
    print("没有可用的 session")
    target_session_id = None

查看 session: 20260410_113250_ef3a2e


In [8]:
# Session 元数据详情
if target_session_id:
    cursor.execute("SELECT * FROM sessions WHERE id = ?", (target_session_id,))
    session = cursor.fetchone()
    
    if session:
        print("Session 元数据:")
        print("=" * 60)
        for key in session.keys():
            value = session[key]
            if key in ['started_at', 'ended_at'] and value:
                value = datetime.fromtimestamp(value).strftime('%Y-%m-%d %H:%M:%S')
            elif key in ['model_config'] and value:
                try:
                    value = json.loads(value)
                except:
                    pass
            print(f"{key:25}: {value}")

Session 元数据:
id                       : 20260410_113250_ef3a2e
source                   : cli
user_id                  : None
model                    : glm-5
model_config             : {'max_iterations': 90, 'reasoning_config': None, 'max_tokens': None}
system_prompt            : You are Hermes Agent, an intelligent AI assistant created by Nous Research. You are helpful, knowledgeable, and direct. You assist users with a wide range of tasks including answering questions, writing and editing code, analyzing information, creative work, and executing actions via your tools. You communicate clearly, admit uncertainty when appropriate, and prioritize being genuinely useful over being verbose unless otherwise directed below. Be targeted and efficient in your exploration and investigations.

You have persistent memory across sessions. Save durable facts using the memory tool: user preferences, environment details, tool quirks, and stable conventions. Memory is injected into every turn, so ke

In [9]:
# Session 的消息内容
if target_session_id:
    cursor.execute("""
        SELECT 
            id,
            role,
            content,
            tool_call_id,
            tool_calls,
            tool_name,
            timestamp,
            token_count,
            reasoning
        FROM messages 
        WHERE session_id = ? 
        ORDER BY timestamp, id
    """, (target_session_id,))
    
    messages = cursor.fetchall()
    print(f"\n共有 {len(messages)} 条消息\n")
    print("=" * 80)
    
    for i, msg in enumerate(messages, 1):
        ts = datetime.fromtimestamp(msg['timestamp']).strftime('%H:%M:%S') if msg['timestamp'] else 'N/A'
        role = msg['role'].upper()
        print(f"\n[{i}] {role} ({ts})")
        print("-" * 60)
        
        if msg['content']:
            content = msg['content'][:500]
            if len(msg['content']) > 500:
                content += "\n... (truncated)"
            print(content)
        
        if msg['tool_calls']:
            print("\n[Tool Calls]")
            try:
                tool_calls = json.loads(msg['tool_calls'])
                for tc in tool_calls:
                    print(f"  - {tc.get('function', {}).get('name', 'unknown')}")
                    args = tc.get('function', {}).get('arguments', '{}')
                    try:
                        args_obj = json.loads(args) if isinstance(args, str) else args
                        print(f"    args: {json.dumps(args_obj, indent=2)[:200]}")
                    except:
                        print(f"    args: {args[:100]}")
            except Exception as e:
                print(f"  Error parsing tool_calls: {e}")
        
        if msg['tool_name']:
            print(f"\n[Tool Result: {msg['tool_name']}]")
            print(f"  tool_call_id: {msg['tool_call_id']}")
        
        if msg['reasoning']:
            reasoning = msg['reasoning'][:300]
            if len(msg['reasoning']) > 300:
                reasoning += "..."
            print(f"\n[Reasoning] {reasoning}")
        
        print("=" * 80)


共有 2 条消息


[1] USER (11:34:02)
------------------------------------------------------------
hi

[2] ASSISTANT (11:34:02)
------------------------------------------------------------
你好！有什么我可以帮助你的吗？我可以协助你完成各种任务，比如：

- 编写和调试代码
- 搜索和分析信息
- 执行终端命令
- 管理文件和项目
- 回答问题

请告诉我你需要什么帮助。

[Reasoning] 用户只是打了个招呼"hi"。根据用户配置，我需要用中文回复。这是一个简单的问候，我应该友好地回应。


## 4. 搜索消息内容 (FTS5)

使用全文搜索查找包含特定关键词的消息。

In [10]:
# 搜索包含特定关键词的消息
search_keyword = "hello"  # 修改这个关键词

try:
    cursor.execute("""
        SELECT m.id, m.session_id, m.role, m.content, m.timestamp
        FROM messages_fts fts
        JOIN messages m ON fts.rowid = m.id
        WHERE fts.content MATCH ?
        ORDER BY m.timestamp DESC
        LIMIT 10
    """, (search_keyword,))
    
    results = cursor.fetchall()
    print(f"搜索 '{search_keyword}' 找到 {len(results)} 条结果\n")
    
    for r in results:
        ts = datetime.fromtimestamp(r['timestamp']).strftime('%Y-%m-%d %H:%M:%S') if r['timestamp'] else 'N/A'
        print(f"Session: {r['session_id'][:20]}... | {r['role']} | {ts}")
        content = r['content'][:150] if r['content'] else '(no content)'
        print(f"  {content}...")
        print()
except Exception as e:
    print(f"搜索失败: {e}")
    print("注意: FTS5 表可能不存在或为空")

搜索 'hello' 找到 0 条结果



## 5. 统计信息

In [11]:
# 统计信息
print("数据库统计:")
print("=" * 40)

# Session 总数
cursor.execute("SELECT COUNT(*) FROM sessions")
print(f"Sessions 总数: {cursor.fetchone()[0]}")

# Messages 总数
cursor.execute("SELECT COUNT(*) FROM messages")
print(f"Messages 总数: {cursor.fetchone()[0]}")

# 按 source 统计
print("\n按来源统计 Sessions:")
cursor.execute("""
    SELECT source, COUNT(*) as count 
    FROM sessions 
    GROUP BY source 
    ORDER BY count DESC
""")
for row in cursor.fetchall():
    print(f"  {row['source'] or 'unknown':15}: {row['count']}")

# 按 role 统计 Messages
print("\n按角色统计 Messages:")
cursor.execute("""
    SELECT role, COUNT(*) as count 
    FROM messages 
    GROUP BY role 
    ORDER BY count DESC
""")
for row in cursor.fetchall():
    print(f"  {row['role'] or 'unknown':15}: {row['count']}")

# 最近活跃的 sessions
print("\n最近 5 个活跃 Sessions:")
cursor.execute("""
    SELECT id, title, source, started_at
    FROM sessions
    ORDER BY started_at DESC
    LIMIT 5
""")
for row in cursor.fetchall():
    ts = datetime.fromtimestamp(row['started_at']).strftime('%Y-%m-%d %H:%M:%S') if row['started_at'] else 'N/A'
    title = row['title'] or '(untitled)'
    print(f"  {ts} | {row['source'] or 'unknown':10} | {title[:30]}")

数据库统计:
Sessions 总数: 4
Messages 总数: 12

按来源统计 Sessions:
  cli            : 4

按角色统计 Messages:
  user           : 6
  assistant      : 5
  tool           : 1

最近 5 个活跃 Sessions:
  2026-04-10 11:32:55 | cli        | (untitled)
  2026-04-10 11:30:11 | cli        | (untitled)
  2026-04-10 11:28:09 | cli        | (untitled)
  2026-04-10 10:48:52 | cli        | (untitled)


## 6. 导出 Session 为 JSON

将会话内容导出为 JSON 格式，方便分析。

In [12]:
# 导出特定 session 为 JSON
if target_session_id:
    cursor.execute("SELECT * FROM sessions WHERE id = ?", (target_session_id,))
    session_row = cursor.fetchone()
    
    if session_row:
        session_dict = dict(session_row)
        
        # 转换时间戳
        for key in ['started_at', 'ended_at']:
            if session_dict.get(key):
                session_dict[key] = datetime.fromtimestamp(session_dict[key]).isoformat()
        
        # 解析 JSON 字段
        if session_dict.get('model_config'):
            try:
                session_dict['model_config'] = json.loads(session_dict['model_config'])
            except:
                pass
        
        # 获取消息
        cursor.execute("SELECT * FROM messages WHERE session_id = ? ORDER BY timestamp, id", (target_session_id,))
        messages = []
        for msg_row in cursor.fetchall():
            msg_dict = dict(msg_row)
            if msg_dict.get('timestamp'):
                msg_dict['timestamp'] = datetime.fromtimestamp(msg_dict['timestamp']).isoformat()
            if msg_dict.get('tool_calls'):
                try:
                    msg_dict['tool_calls'] = json.loads(msg_dict['tool_calls'])
                except:
                    pass
            messages.append(msg_dict)
        
        session_dict['messages'] = messages
        
        # 显示 JSON (缩进)
        print(json.dumps(session_dict, indent=2, ensure_ascii=False, default=str)[:3000])
        print("\n... (truncated)")
        
        # 可选: 保存到文件
        output_file = f"session_{target_session_id[:8]}.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(session_dict, f, indent=2, ensure_ascii=False, default=str)
        print(f"\n✓ 已保存到: {output_file}")

{
  "id": "20260410_113250_ef3a2e",
  "source": "cli",
  "user_id": null,
  "model": "glm-5",
  "model_config": {
    "max_iterations": 90,
    "reasoning_config": null,
    "max_tokens": null
  },
  "system_prompt": "You are Hermes Agent, an intelligent AI assistant created by Nous Research. You are helpful, knowledgeable, and direct. You assist users with a wide range of tasks including answering questions, writing and editing code, analyzing information, creative work, and executing actions via your tools. You communicate clearly, admit uncertainty when appropriate, and prioritize being genuinely useful over being verbose unless otherwise directed below. Be targeted and efficient in your exploration and investigations.\n\nYou have persistent memory across sessions. Save durable facts using the memory tool: user preferences, environment details, tool quirks, and stable conventions. Memory is injected into every turn, so keep it compact and focused on facts that will still matter late

In [13]:
# 关闭连接
conn.close()
print("✓ 数据库连接已关闭")

✓ 数据库连接已关闭
